# Watermark Removal using Convolutional Autoencoder

This notebook demonstrates how to train a convolutional autoencoder to remove watermarks from images using the **Watermark Removal Image Pairs Dataset**.

## Dataset
- **802 image pairs** (watermarked + clean)
- **Train**: 641 pairs (80%)
- **Valid**: 80 pairs (10%)
- **Test**: 81 pairs (10%)
- **Resolution**: 512×512 pixels

## Model Architecture
Convolutional Autoencoder with:
- **Encoder**: 3 blocks (64, 128, 256 filters)
- **Bottleneck**: 512 filters
- **Decoder**: 3 blocks (256, 128, 64 filters)
- **Total parameters**: ~26M

## 1. Setup and Imports

In [ ]:
import os
import json
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau, TensorBoard

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {tf.config.list_physical_devices('GPU')}")

## 2. Configuration

In [ ]:
# Configuration
CONFIG = {
    'dataset_path': '/kaggle/input/watermark-removal-image-pairs/wm-nown',
    'batch_size': 16,  # Adjust based on GPU memory
    'image_size': (512, 512),
    'epochs': 50,
    'learning_rate': 0.001,
    'random_seed': 42,
    'augment_train': True,
}

# Set random seeds
np.random.seed(CONFIG['random_seed'])
tf.random.set_seed(CONFIG['random_seed'])

print("Configuration:")
for key, value in CONFIG.items():
    print(f"  {key}: {value}")

## 3. Explore Dataset

In [ ]:
# Check dataset structure
dataset_path = Path(CONFIG['dataset_path'])

print("Dataset structure:")
for split in ['train', 'valid', 'test']:
    wm_dir = dataset_path / split / 'watermark'
    clean_dir = dataset_path / split / 'no-watermark'
    
    wm_count = len(list(wm_dir.glob('*.jpg')))
    clean_count = len(list(clean_dir.glob('*.jpg')))
    
    print(f"\n{split.upper()}:")
    print(f"  Watermarked: {wm_count}")
    print(f"  Clean: {clean_count}")

# Load dataset statistics
stats_file = dataset_path / 'dataset_stats.json'
if stats_file.exists():
    with open(stats_file, 'r') as f:
        stats = json.load(f)
    print("\nDataset Statistics:")
    print(json.dumps(stats['dataset_info'], indent=2))

## 4. Visualize Sample Images

In [ ]:
import random
from PIL import Image

# Get sample images
train_wm_dir = dataset_path / 'train' / 'watermark'
train_clean_dir = dataset_path / 'train' / 'no-watermark'

sample_files = random.sample(list(train_wm_dir.glob('*.jpg')), 3)

fig, axes = plt.subplots(3, 2, figsize=(12, 15))

for idx, wm_path in enumerate(sample_files):
    # Load images
    wm_img = Image.open(wm_path)
    clean_img = Image.open(train_clean_dir / wm_path.name)
    
    # Display
    axes[idx, 0].imshow(wm_img)
    axes[idx, 0].set_title(f'Watermarked - {wm_path.name}')
    axes[idx, 0].axis('off')
    
    axes[idx, 1].imshow(clean_img)
    axes[idx, 1].set_title(f'Clean (Target)')
    axes[idx, 1].axis('off')

plt.tight_layout()
plt.show()

## 5. Create Data Pipeline

In [ ]:
def load_image_pair(watermark_path, clean_path):
    """
    Load and preprocess a pair of images
    """
    # Load watermarked image
    watermark_img = tf.io.read_file(watermark_path)
    watermark_img = tf.image.decode_jpeg(watermark_img, channels=3)
    watermark_img = tf.cast(watermark_img, tf.float32) / 255.0
    
    # Load clean image
    clean_img = tf.io.read_file(clean_path)
    clean_img = tf.image.decode_jpeg(clean_img, channels=3)
    clean_img = tf.cast(clean_img, tf.float32) / 255.0
    
    return watermark_img, clean_img


def augment_images(watermark_img, clean_img):
    """
    Apply data augmentation to both images
    IMPORTANT: Same augmentation must be applied to both!
    """
    # Random horizontal flip
    if tf.random.uniform(()) > 0.5:
        watermark_img = tf.image.flip_left_right(watermark_img)
        clean_img = tf.image.flip_left_right(clean_img)
    
    # Random vertical flip
    if tf.random.uniform(()) > 0.5:
        watermark_img = tf.image.flip_up_down(watermark_img)
        clean_img = tf.image.flip_up_down(clean_img)
    
    # Random brightness
    brightness_delta = tf.random.uniform((), -0.1, 0.1)
    watermark_img = tf.image.adjust_brightness(watermark_img, brightness_delta)
    clean_img = tf.image.adjust_brightness(clean_img, brightness_delta)
    
    # Random contrast
    contrast_factor = tf.random.uniform((), 0.9, 1.1)
    watermark_img = tf.image.adjust_contrast(watermark_img, contrast_factor)
    clean_img = tf.image.adjust_contrast(clean_img, contrast_factor)
    
    # Clip to valid range
    watermark_img = tf.clip_by_value(watermark_img, 0.0, 1.0)
    clean_img = tf.clip_by_value(clean_img, 0.0, 1.0)
    
    return watermark_img, clean_img


def create_dataset(split, shuffle=True, augment=False):
    """
    Create TensorFlow dataset for specified split
    """
    watermark_dir = dataset_path / split / 'watermark'
    clean_dir = dataset_path / split / 'no-watermark'
    
    # Get all image paths
    watermark_paths = sorted([str(p) for p in watermark_dir.glob('*.jpg')])
    clean_paths = sorted([str(p) for p in clean_dir.glob('*.jpg')])
    
    print(f"Creating {split} dataset: {len(watermark_paths)} pairs")
    
    # Create dataset
    ds = tf.data.Dataset.from_tensor_slices((watermark_paths, clean_paths))
    
    # Shuffle
    if shuffle:
        ds = ds.shuffle(buffer_size=len(watermark_paths), seed=CONFIG['random_seed'])
    
    # Load images
    ds = ds.map(load_image_pair, num_parallel_calls=tf.data.AUTOTUNE)
    
    # Augment
    if augment:
        ds = ds.map(augment_images, num_parallel_calls=tf.data.AUTOTUNE)
    
    # Batch and prefetch
    ds = ds.batch(CONFIG['batch_size'])
    ds = ds.prefetch(tf.data.AUTOTUNE)
    
    return ds


# Create datasets
print("Creating datasets...\n")
train_ds = create_dataset('train', shuffle=True, augment=CONFIG['augment_train'])
valid_ds = create_dataset('valid', shuffle=False, augment=False)
test_ds = create_dataset('test', shuffle=False, augment=False)

print("\nDataset pipeline created successfully!")

## 6. Test Data Pipeline

In [ ]:
# Test the data pipeline
for watermark_batch, clean_batch in train_ds.take(1):
    print(f"Watermark batch shape: {watermark_batch.shape}")
    print(f"Clean batch shape: {clean_batch.shape}")
    print(f"Value range: [{tf.reduce_min(watermark_batch):.3f}, {tf.reduce_max(watermark_batch):.3f}]")
    
    # Visualize batch samples
    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    
    for i in range(min(4, watermark_batch.shape[0])):
        axes[0, i].imshow(watermark_batch[i])
        axes[0, i].set_title(f'Watermarked {i+1}')
        axes[0, i].axis('off')
        
        axes[1, i].imshow(clean_batch[i])
        axes[1, i].set_title(f'Clean {i+1}')
        axes[1, i].axis('off')
    
    plt.tight_layout()
    plt.show()

print("\nData pipeline is working correctly!")

## 7. Build Autoencoder Model

In [ ]:
def build_autoencoder(input_shape=(512, 512, 3)):
    """
    Build convolutional autoencoder for watermark removal
    """
    inputs = layers.Input(shape=input_shape, name='watermarked_input')
    
    # ========== ENCODER ==========
    
    # Encoder block 1 (512x512 -> 256x256)
    x = layers.Conv2D(64, 3, activation='relu', padding='same', name='enc_conv1_1')(inputs)
    x = layers.Conv2D(64, 3, activation='relu', padding='same', name='enc_conv1_2')(x)
    x = layers.MaxPooling2D(2, padding='same', name='enc_pool1')(x)
    
    # Encoder block 2 (256x256 -> 128x128)
    x = layers.Conv2D(128, 3, activation='relu', padding='same', name='enc_conv2_1')(x)
    x = layers.Conv2D(128, 3, activation='relu', padding='same', name='enc_conv2_2')(x)
    x = layers.MaxPooling2D(2, padding='same', name='enc_pool2')(x)
    
    # Encoder block 3 (128x128 -> 64x64)
    x = layers.Conv2D(256, 3, activation='relu', padding='same', name='enc_conv3_1')(x)
    x = layers.Conv2D(256, 3, activation='relu', padding='same', name='enc_conv3_2')(x)
    x = layers.MaxPooling2D(2, padding='same', name='enc_pool3')(x)
    
    # ========== BOTTLENECK ==========
    
    x = layers.Conv2D(512, 3, activation='relu', padding='same', name='bottleneck_conv1')(x)
    x = layers.Conv2D(512, 3, activation='relu', padding='same', name='bottleneck_conv2')(x)
    
    # ========== DECODER ==========
    
    # Decoder block 1 (64x64 -> 128x128)
    x = layers.Conv2DTranspose(256, 2, strides=2, padding='same', name='dec_upsample1')(x)
    x = layers.Conv2D(256, 3, activation='relu', padding='same', name='dec_conv1_1')(x)
    x = layers.Conv2D(256, 3, activation='relu', padding='same', name='dec_conv1_2')(x)
    
    # Decoder block 2 (128x128 -> 256x256)
    x = layers.Conv2DTranspose(128, 2, strides=2, padding='same', name='dec_upsample2')(x)
    x = layers.Conv2D(128, 3, activation='relu', padding='same', name='dec_conv2_1')(x)
    x = layers.Conv2D(128, 3, activation='relu', padding='same', name='dec_conv2_2')(x)
    
    # Decoder block 3 (256x256 -> 512x512)
    x = layers.Conv2DTranspose(64, 2, strides=2, padding='same', name='dec_upsample3')(x)
    x = layers.Conv2D(64, 3, activation='relu', padding='same', name='dec_conv3_1')(x)
    x = layers.Conv2D(64, 3, activation='relu', padding='same', name='dec_conv3_2')(x)
    
    # ========== OUTPUT ==========
    
    outputs = layers.Conv2D(3, 3, activation='sigmoid', padding='same', name='clean_output')(x)
    
    # Create model
    model = models.Model(inputs=inputs, outputs=outputs, name='watermark_remover')
    
    return model


# Build model
print("Building model...")
model = build_autoencoder(input_shape=(512, 512, 3))

# Compile model
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=CONFIG['learning_rate']),
    loss='mse',
    metrics=['mae', keras.metrics.MeanSquaredError(name='mse')]
)

print("\nModel Summary:")
model.summary()

## 8. Setup Callbacks

In [ ]:
# Create callbacks
callbacks = [
    ModelCheckpoint(
        'watermark_remover_best.keras',
        save_best_only=True,
        monitor='val_loss',
        mode='min',
        verbose=1
    ),
    
    EarlyStopping(
        monitor='val_loss',
        patience=10,
        restore_best_weights=True,
        verbose=1
    ),
    
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=5,
        min_lr=1e-7,
        verbose=1
    ),
    
    TensorBoard(
        log_dir='./logs',
        histogram_freq=1
    )
]

print("Callbacks configured:")
for cb in callbacks:
    print(f"  - {cb.__class__.__name__}")

## 9. Train Model

In [ ]:
# Train model
print("Starting training...\n")

history = model.fit(
    train_ds,
    validation_data=valid_ds,
    epochs=CONFIG['epochs'],
    callbacks=callbacks,
    verbose=1
)

print("\nTraining complete!")

## 10. Plot Training History

In [ ]:
# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss
axes[0].plot(history.history['loss'], label='Train Loss')
axes[0].plot(history.history['val_loss'], label='Val Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss (MSE)')
axes[0].set_title('Training and Validation Loss')
axes[0].legend()
axes[0].grid(True)

# MAE
axes[1].plot(history.history['mae'], label='Train MAE')
axes[1].plot(history.history['val_mae'], label='Val MAE')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('MAE')
axes[1].set_title('Training and Validation MAE')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

# Print final metrics
print("\nFinal Metrics:")
print(f"  Train Loss: {history.history['loss'][-1]:.6f}")
print(f"  Val Loss: {history.history['val_loss'][-1]:.6f}")
print(f"  Train MAE: {history.history['mae'][-1]:.6f}")
print(f"  Val MAE: {history.history['val_mae'][-1]:.6f}")

## 11. Evaluate on Test Set

In [ ]:
# Load best model
print("Loading best model...")
best_model = keras.models.load_model('watermark_remover_best.keras')

# Evaluate on test set
print("\nEvaluating on test set...")
test_results = best_model.evaluate(test_ds, verbose=1)

print("\nTest Set Results:")
for metric_name, value in zip(best_model.metrics_names, test_results):
    print(f"  {metric_name}: {value:.6f}")

## 12. Visualize Predictions

In [ ]:
# Get test samples
for watermark_batch, clean_batch in test_ds.take(1):
    predictions = best_model.predict(watermark_batch)
    
    # Visualize
    num_samples = min(4, watermark_batch.shape[0])
    fig, axes = plt.subplots(num_samples, 3, figsize=(15, 5*num_samples))
    
    if num_samples == 1:
        axes = [axes]
    
    for i in range(num_samples):
        # Watermarked
        axes[i][0].imshow(watermark_batch[i])
        axes[i][0].set_title('Input (Watermarked)')
        axes[i][0].axis('off')
        
        # Predicted (cleaned)
        axes[i][1].imshow(predictions[i])
        axes[i][1].set_title('Output (Predicted Clean)')
        axes[i][1].axis('off')
        
        # Ground truth
        axes[i][2].imshow(clean_batch[i])
        axes[i][2].set_title('Target (Ground Truth)')
        axes[i][2].axis('off')
    
    plt.tight_layout()
    plt.show()

print("\nPredictions visualized!")

## 13. Calculate PSNR and SSIM

In [ ]:
# Calculate PSNR and SSIM on test set
psnr_values = []
ssim_values = []

for watermark_batch, clean_batch in test_ds:
    predictions = best_model.predict(watermark_batch, verbose=0)
    
    for i in range(predictions.shape[0]):
        # PSNR
        psnr = tf.image.psnr(clean_batch[i], predictions[i], max_val=1.0)
        psnr_values.append(psnr.numpy())
        
        # SSIM
        ssim = tf.image.ssim(clean_batch[i], predictions[i], max_val=1.0)
        ssim_values.append(ssim.numpy())

print("\nImage Quality Metrics:")
print(f"  Average PSNR: {np.mean(psnr_values):.2f} dB")
print(f"  Average SSIM: {np.mean(ssim_values):.4f}")
print(f"  Min PSNR: {np.min(psnr_values):.2f} dB")
print(f"  Max PSNR: {np.max(psnr_values):.2f} dB")
print(f"  Min SSIM: {np.min(ssim_values):.4f}")
print(f"  Max SSIM: {np.max(ssim_values):.4f}")

## 14. Save Final Model

In [ ]:
# Save final model
best_model.save('watermark_remover_final.keras')
print("Model saved as: watermark_remover_final.keras")

# Save model in SavedModel format for deployment
best_model.save('watermark_remover_savedmodel', save_format='tf')
print("Model saved in SavedModel format: watermark_remover_savedmodel/")

print("\nAll done! Model is ready for deployment.")

## Summary

### Model Performance
- Successfully trained a convolutional autoencoder for watermark removal
- Model learns to reconstruct clean images from watermarked inputs

### Next Steps
1. **Improve Results**: Try U-Net architecture with skip connections
2. **Better Loss**: Implement perceptual loss (VGG-based) for visual quality
3. **Train Longer**: Increase epochs to 100+ with EarlyStopping
4. **GAN Approach**: Try Pix2Pix or CycleGAN for better results
5. **Deploy**: Use the saved model for inference on new images

### Files Generated
- `watermark_remover_best.keras` - Best model during training
- `watermark_remover_final.keras` - Final model
- `watermark_remover_savedmodel/` - Deployment-ready format
- `logs/` - TensorBoard logs